# 02 - Exploratory data analysis
Uses `reports/eda_summary.json` (produced by `scripts/run_eda.py` from **train + validation rows only**, so the test set stays unseen) and the processed data. Charts are tied to a question; interpret them in the last cell.

**Questions:** How does severity change with speed limit, road type, lighting, weather, surface, urban/rural and time? Which features are most associated with severity (effect size, not p-value)?

In [ ]:
import os, sys, json
from pathlib import Path
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
ROOT = Path(os.environ.get("ROADRISK_ROOT") or Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.insert(0, str(ROOT))
from ml.config import get_settings
from ml.feature_config import CLASS_NAMES
cfg = get_settings()
pd.set_option("display.max_columns", 50, "display.width", 140)
def load_report(name):
    p = cfg.reports_dir / name
    if not p.exists():
        raise FileNotFoundError(f"{p} not found - run the pipeline scripts first (see README).")
    return json.loads(p.read_text(encoding="utf-8"))
COLORS = {"Slight": "#5b8def", "Serious": "#f2a33a", "Fatal": "#d64545"}

In [ ]:
eda = load_report("eda_summary.json")
print("Rows analysed:", f"{eda['rows_analysed']:,}", "| excluded:", eda["excluded_from_eda"])
print("Headline stats:", eda["headline_stats"])
pd.Series(eda["target_distribution"]).to_frame("rows")

## Which features are associated with severity? (Cramér's V = effect size)

In [ ]:
assoc = pd.Series({k: v["cramers_v"] for k, v in eda["association_with_severity"].items()}).sort_values()
assoc.plot.barh(figsize=(7, 6), color="#4a6fa5", title="Association with severity (Cramér's V)"); plt.tight_layout(); plt.show()

## Severity mix by feature (% of collisions within each level)

In [ ]:
def plot_rates(feature, order=None):
    t = pd.DataFrame(eda["severity_by_feature"][feature]).T
    t = t.loc[order] if order else t
    ax = t[[f"pct_{c}" for c in CLASS_NAMES]].rename(columns=lambda c: c[4:]).plot.bar(stacked=True, figsize=(8, 3.2),
          color=[COLORS[c] for c in CLASS_NAMES], title=f"Severity mix by {feature}")
    ax.set_ylabel("% of collisions"); ax.legend(loc="upper right", bbox_to_anchor=(1.18, 1)); plt.tight_layout(); plt.show()
    return t[["n"] + [f"pct_{c}" for c in CLASS_NAMES]]
for f in ["speed_limit", "road_type_group", "lighting_group", "weather_group", "surface_group", "urban_or_rural_area", "junction_group"]:
    if f in eda["severity_by_feature"]:
        display(plot_rates(f))

## Severity vs time

In [ ]:
for f in ["hour", "weekday", "month", "is_night"]:
    if f in eda["severity_by_feature"]:
        t = pd.DataFrame(eda["severity_by_feature"][f]).T
        t.index = t.index.astype(float).astype(int) if f != "is_night" else t.index
        fig, ax = plt.subplots(1, 2, figsize=(10, 3))
        t["n"].plot(ax=ax[0], title=f"Collisions by {f}", marker="o")
        t[["pct_Serious", "pct_Fatal"]].plot(ax=ax[1], title=f"Serious / Fatal share by {f} (%)", marker="o", color=["#f2a33a", "#d64545"])
        plt.tight_layout(); plt.show()

## Trend over time

In [ ]:
if "monthly_counts" in eda:
    m = pd.DataFrame(eda["monthly_counts"]).T[CLASS_NAMES]; m.plot(figsize=(10, 3), color=[COLORS[c] for c in CLASS_NAMES], title="Monthly collisions by severity"); plt.show()
if "yearly_counts" in eda:
    display(pd.DataFrame(eda["yearly_counts"]).T)

## Numeric distributions

In [ ]:
pd.DataFrame(eda["numeric_distributions"])

## Your interpretation
_After running on your data, write what the charts show (and what they do NOT show: association is not causation; fatal collisions are rare, so percentages in small groups are noisy)._